# Main Script as Jupyter Notebook

This notebook is a conversion of the `main.py` script, designed for easier debugging by running cell by cell.

In [ ]:
# Imports and Initial Setup
import os 
import argparse # Kept for reference, but parameters will be set manually
import configparser
import pandas as pd
import torch
import pytorch_lightning as pl


from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objs as go
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from pynas.core.population import Population
from datasets.RawVessels.loader import RawVesselsDataModule, RawVesselsDataset

import numpy as np
import cv2
# Additional imports for statistical threshold calculation
from scipy.optimize import minimize
from matplotlib import pyplot as plt
from scipy.ndimage import gaussian_filter, median_filter
from scipy.signal import hilbert
from skimage import io
from skimage.restoration import denoise_tv_chambolle
from skimage.feature import blob_log, peak_local_max

# Pandas display option
pd.set_option('display.max_colwidth', None)
# PyTorch Lightning seed and precision
pl.seed_everything(seed=42, workers=True) # Example seed, change as needed
torch.set_float32_matmul_precision("medium")

# Utils

In [ ]:
import numpy as np

def extract_dominant_phase_components(phase_matrix: np.ndarray, N: int):
    """
    Extract the N dominant frequency components from a wrapped SAR phase matrix using FFT.

    Args:
        phase_matrix (np.ndarray): 2D wrapped phase matrix (real-valued, in [-π, π]).
        N (int): Number of dominant frequency components to extract.

    Returns:
        List[Tuple[Tuple[int, int], complex, np.ndarray]]: List of N tuples:
            - ((fx, fy), amplitude, spatial_pattern) where fx, fy are frequency indices,
              amplitude is the complex FFT coefficient, and spatial_pattern is the
              reconstructed phase matrix for that component.
    """
    if phase_matrix.ndim != 2:
        raise ValueError("Input phase_matrix must be 2D.")

    # Compute the 2D FFT
    fft_matrix = np.fft.fftshift(np.fft.fft2(phase_matrix))

    # Get the flattened index list sorted by magnitude (descending)
    magnitude = np.abs(fft_matrix)
    flat_indices = np.argsort(magnitude.ravel())[::-1]

    # Map flat indices to 2D frequency indices
    dominant_components = []
    rows, cols = phase_matrix.shape
    center_r, center_c = rows // 2, cols // 2

    for idx in flat_indices[:N]:
        fy, fx = np.unravel_index(idx, fft_matrix.shape)
        amplitude = fft_matrix[fy, fx]
        # Shift to frequency coordinates relative to center (0 frequency at center)
        fx_shifted = fx - center_c
        fy_shifted = fy - center_r

        # Create a 2D array with only this frequency component (in frequency domain)
        fft_component = np.zeros_like(fft_matrix, dtype=complex)
        fft_component[fy, fx] = amplitude
        # Inverse FFT to get spatial pattern
        spatial_pattern = np.fft.ifft2(np.fft.ifftshift(fft_component)).real

        dominant_components.append(((fx_shifted, fy_shifted), amplitude, spatial_pattern))

    return dominant_components

# Data Loading

In [ ]:
# ----- Load the dataset ------
root_dir_datamodule = '/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/data/TASI/DataSAR_real_refined'
# Ensure the root directory exists
if not Path(root_dir_datamodule).exists():
    raise FileNotFoundError(f"Root directory {root_dir_datamodule} does not exist.")
# Load image and mask paths
root_dir_datamodule = Path(root_dir_datamodule)
if not (root_dir_datamodule / 'inputs').exists() or not (root_dir_datamodule / 'masks').exists():
    raise FileNotFoundError(f"Expected directories 'inputs' and 'masks' not found in {root_dir_datamodule}.")

image_paths = [x for x in (root_dir_datamodule / 'inputs').glob('*.pkl')]
mask_paths = [x for x in (root_dir_datamodule / 'masks').glob('*.pkl')]


loader = RawVesselsDataset(image_paths, 
                           mask_paths, 
                           transform=None)


In [ ]:

# ----- Load the dataset ------
# Ensure the dataset is loaded correctly
# Try loading and inspecting a sample
rand_idx = 5  # Change this index to load different samples
sample = loader[rand_idx]
img, mask = sample 


Re, Im = img  # Real and Imaginary parts of the complex image
# Make Amplitude and Phase
amp = np.abs(Re + 1j * Im)  # Amplitude
phase = np.angle(Re + 1j * Im)  # Phase

# Compute mean and std for phase
mean = np.mean(phase)
std = np.std(phase)
vmin = mean - 0.25 * std
vmax = mean + 0.25 * std

fig, axs = plt.subplots(1, 3, figsize=(14, 7), dpi=140)

axs[0].imshow(amp, cmap='gray')
axs[0].set_title('Amplitude')
axs[0].axis('off')

# Increased contrast for phase
# Apply Gaussian smoothing to the phase before displaying
phase_smoothed = gaussian_filter(phase, sigma=3.0)
axs[1].imshow(phase_smoothed, cmap='inferno', vmin=vmin, vmax=vmax)
axs[1].set_title('Phase (Smoothed, Increased Contrast)')
axs[1].axis('off')

axs[2].imshow(mask, cmap='gray')
axs[2].set_title('Ground Truth Mask')
axs[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter
from skimage.morphology import white_tophat, disk

# ----- Load the dataset ------
# Ensure the dataset is loaded correctly
# Try loading and inspecting a sample
rand_idx = 1  # Change this index to load different samples
sample = loader[rand_idx]
img, mask = sample 


Re, Im = img  # Real and Imaginary parts of the complex image
# Make Amplitude and Phase
amp = np.abs(Re + 1j * Im)  # Amplitude
phase = np.angle(Re + 1j * Im)  # Phase

# Compute mean and std for phase
mean = np.mean(phase)
std = np.std(phase)
vmin = mean - 0.5 * std
vmax = mean + 0.5 * std

fig, axs = plt.subplots(1, 2, figsize=(14, 7), dpi=140)

# Increased contrast for phase
# Apply Gaussian smoothing to the phase before displaying
phase_smoothed = gaussian_filter(phase, sigma=4.0)
# Apply white top-hat to remove small objects (disk radius can be adjusted)
phase_tophat = white_tophat(phase_smoothed, footprint=disk(20))
axs[0].imshow(phase_tophat, cmap='inferno', vmin=vmin, vmax=vmax)
axs[0].set_title('Phase (Smoothed + White Top-Hat)')
axs[0].axis('off')

axs[1].imshow(mask, cmap='gray')
axs[1].set_title('Ground Truth Mask')
axs[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(phase_smoothed.ravel(), bins=100, color='purple', alpha=0.7)
plt.title("Histogram of phase_smoothed")
plt.xlabel("Phase Value")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

In [ ]:
def calculate_adaptive_thresholds(phase_image, sigma=1.5, ratio=2.0, percentile_low=50, percentile_high=85):
    """
    Calculate adaptive Canny edge detection thresholds based on phase image statistics.
    
    Args:
        phase_image (np.ndarray): Phase image array
        sigma (float): Gaussian smoothing parameter for gradient calculation
        ratio (float): Ratio between high and low thresholds (typically 2-3)
        percentile_low (float): Percentile for low threshold (0-100)
        percentile_high (float): Percentile for high threshold (0-100)
        
    Returns:
        tuple: (threshold1, threshold2) - lower and upper thresholds for Canny
    """
    from scipy.ndimage import gaussian_filter
    
    # Apply smoothing to reduce noise
    smoothed_phase = gaussian_filter(phase_image, sigma=sigma)
    
    # Normalize to 0-255 range for gradient calculation
    phase_norm = ((smoothed_phase - smoothed_phase.min()) / 
                  (smoothed_phase.max() - smoothed_phase.min()) * 255).astype(np.uint8)
    
    # Calculate gradients using Sobel operators
    grad_x = cv2.Sobel(phase_norm, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(phase_norm, cv2.CV_64F, 0, 1, ksize=3)
    
    # Calculate gradient magnitude
    gradient_magnitude = np.sqrt(grad_x**2 + grad_y**2)
    
    # Use percentiles of gradient magnitude to set thresholds
    threshold1 = np.percentile(gradient_magnitude, percentile_low)
    threshold2 = np.percentile(gradient_magnitude, percentile_high)
    
    # Ensure proper ratio between thresholds
    if threshold2 / threshold1 < ratio:
        threshold2 = threshold1 * ratio
    
    # Convert to integers and ensure reasonable bounds
    threshold1 = max(10, int(threshold1))  # Minimum threshold of 10
    threshold2 = max(threshold1 * ratio, int(threshold2))  # Ensure ratio is maintained
    threshold2 = min(255, threshold2)  # Maximum threshold of 255
    
    return threshold1, threshold2

print("Adaptive threshold calculation function defined successfully!")

In [ ]:
from scipy.optimize import minimize

# Improved Hyperbola-based SAR Fringe Analysis Pipeline with Adaptive Thresholds
def hyperbola_sar_analysis_adaptive(real_part, imaginary_part, 
                                   threshold1=None, threshold2=None, 
                                   lambda_=2*np.pi, visualize=True, 
                                   adaptive_thresholds=True):
    """
    SAR fringe analysis using hyperbola fitting for scatterer detection with adaptive thresholds.
    
    Args:
        real_part (np.ndarray): Real component of complex SAR image
        imaginary_part (np.ndarray): Imaginary component of complex SAR image
        threshold1 (int): Lower threshold for Canny edge detection (auto-calculated if None)
        threshold2 (int): Upper threshold for Canny edge detection (auto-calculated if None)
        lambda_ (float): Wavelength parameter for hyperbola model
        visualize (bool): Whether to display results
        adaptive_thresholds (bool): Use statistical calculation for thresholds
        
    Returns:
        tuple: (scatter_location, processed_image, thresholds_used)
    """
    # Create phase image
    complex_image = real_part + 1j * imaginary_part
    phase_image = np.angle(complex_image)
    
    # Apply Gaussian smoothing to reduce noise before edge detection
    smoothed_phase = gaussian_filter(phase_image, sigma=1.5)
    
    # Calculate adaptive thresholds if not provided
    if adaptive_thresholds or threshold1 is None or threshold2 is None:
        calc_thresh1, calc_thresh2 = calculate_adaptive_thresholds(phase_image)
        if threshold1 is None:
            threshold1 = calc_thresh1
        if threshold2 is None:
            threshold2 = calc_thresh2
        print(f"Using adaptive thresholds: threshold1={threshold1}, threshold2={threshold2}")
    else:
        print(f"Using provided thresholds: threshold1={threshold1}, threshold2={threshold2}")
    
    # Better normalization for edge detection
    phase_norm = ((smoothed_phase - smoothed_phase.min()) / 
                  (smoothed_phase.max() - smoothed_phase.min()) * 255).astype(np.uint8)
    
    # Apply additional smoothing to the normalized image
    phase_norm = cv2.GaussianBlur(phase_norm, (5, 5), 0)
    
    # Edge detection with calculated thresholds
    edges = cv2.Canny(phase_norm, threshold1//2, threshold2//2)
    
    # Apply morphological operations to clean up edges
    kernel = np.ones((3,3), np.uint8)
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
    edges = cv2.morphologyEx(edges, cv2.MORPH_OPEN, kernel)
    
    # Extract fringe contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Filter contours by minimum area
    min_area = 50
    contours = [c for c in contours if cv2.contourArea(c) > min_area]
    
    if len(contours) == 0:
        print("No significant contours found. Try adjusting threshold parameters.")
        # Use image center as fallback
        center_x, center_y = phase_image.shape[1]//2, phase_image.shape[0]//2
        return (center_x, center_y), phase_norm, (threshold1, threshold2)
    
    # Use largest contour
    largest_contour = max(contours, key=cv2.contourArea)
    contour_points = largest_contour.reshape(-1, 2)
    
    # Define hyperbola model
    def hyperbola_model(params, points, lambda_):
        x_s, y_s = params
        x_0, y_0 = phase_image.shape[1]//2, phase_image.shape[0]//2  # Reference point
        residuals = []
        for (x, y) in points:
            d = np.sqrt((x - x_s)**2 + (y - y_s)**2) - np.sqrt((x - x_0)**2 + (y - y_0)**2)
            residuals.append((d % lambda_)**2)
        return np.sum(residuals)
    
    # Initial guess (use center if previous analysis not available)
    try:
        init_x_val, init_y_val = refined_x, refined_y
    except NameError:
        init_x_val, init_y_val = phase_image.shape[1]//2, phase_image.shape[0]//2
    
    # Fit hyperbola model to contour points
    try:
        result = minimize(hyperbola_model, x0=[init_x_val, init_y_val], 
                         args=(contour_points, lambda_),
                         method='L-BFGS-B')
        scatter_location = result.x
    except:
        print("Optimization failed, using initial guess")
        scatter_location = [init_x_val, init_y_val]
    
    if visualize:
        plt.figure(figsize=(20, 6))
        
        plt.subplot(1, 5, 1)
        plt.imshow(phase_norm, cmap='gray')
        plt.title('Normalized Phase (Smoothed)')
        plt.axis('off')
        
        plt.subplot(1, 5, 2)
        plt.imshow(edges, cmap='gray')
        plt.title(f'Edge Detection\n(T1={threshold1//2}, T2={threshold2//2})')
        plt.axis('off')
        
        plt.subplot(1, 5, 3)
        # Show threshold statistics
        grad_x = cv2.Sobel(phase_norm, cv2.CV_64F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(phase_norm, cv2.CV_64F, 0, 1, ksize=3)
        gradient_magnitude = np.sqrt(grad_x**2 + grad_y**2)
        plt.hist(gradient_magnitude.ravel(), bins=50, alpha=0.7, color='blue')
        plt.axvline(threshold1, color='red', linestyle='--', label=f'T1={threshold1}')
        plt.axvline(threshold2, color='orange', linestyle='--', label=f'T2={threshold2}')
        plt.title('Gradient Magnitude Distribution')
        plt.xlabel('Gradient Magnitude')
        plt.ylabel('Frequency')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 5, 4)
        # Show edge quality metrics
        edge_density = np.sum(edges > 0) / edges.size
        num_labels, labels = cv2.connectedComponents(edges)
        plt.text(0.1, 0.8, f'Edge Density: {edge_density:.4f}', transform=plt.gca().transAxes, fontsize=12)
        plt.text(0.1, 0.7, f'Components: {num_labels-1}', transform=plt.gca().transAxes, fontsize=12)
        plt.text(0.1, 0.6, f'Threshold Ratio: {threshold2/threshold1:.2f}', transform=plt.gca().transAxes, fontsize=12)
        plt.text(0.1, 0.5, f'Mean Gradient: {np.mean(gradient_magnitude):.1f}', transform=plt.gca().transAxes, fontsize=12)
        plt.text(0.1, 0.4, f'Std Gradient: {np.std(gradient_magnitude):.1f}', transform=plt.gca().transAxes, fontsize=12)
        plt.title('Statistics')
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        plt.axis('off')
        
        plt.subplot(1, 5, 5)
        plt.imshow(phase_norm, cmap='gray')
        if len(contours) > 0:
            # Create a copy for drawing
            display_img = phase_norm.copy()
            cv2.drawContours(display_img, [largest_contour], -1, 128, 2)
            plt.imshow(display_img, cmap='gray')
        plt.scatter(scatter_location[0], scatter_location[1], 
                   c='red', s=100, marker='x', linewidths=3, label='Hyperbola Fit')
        plt.scatter(init_x_val, init_y_val, c='lime', s=100, marker='o', 
                   linewidths=3, label='Initial Guess')
        plt.title('Hyperbola-based Scatterer Detection')
        plt.legend()
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
    
    return tuple(scatter_location), phase_norm, (threshold1, threshold2)

print("Improved hyperbola analysis function with adaptive thresholds defined successfully!")

In [ ]:
# Comprehensive threshold analysis using different statistical methods

def analyze_threshold_methods(real_part, imaginary_part, visualize=True):
    """
    Analyze different statistical methods for calculating Canny thresholds.
    """
    # Create phase image
    complex_image = real_part + 1j * imaginary_part
    phase_image = np.angle(complex_image)
    
    # Apply smoothing
    smoothed_phase = gaussian_filter(phase_image, sigma=1.5)
    
    # Normalize phase
    phase_norm = ((smoothed_phase - smoothed_phase.min()) / 
                  (smoothed_phase.max() - smoothed_phase.min()) * 255).astype(np.uint8)
    
    # Calculate gradients
    grad_x = cv2.Sobel(phase_norm, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(phase_norm, cv2.CV_64F, 0, 1, ksize=3)
    gradient_magnitude = np.sqrt(grad_x**2 + grad_y**2)
    
    # Method 1: Percentile-based (current approach)
    thresh1_p50 = np.percentile(gradient_magnitude, 50)
    thresh2_p85 = np.percentile(gradient_magnitude, 85)
    
    # Method 2: Mean + std deviation
    grad_mean = np.mean(gradient_magnitude)
    grad_std = np.std(gradient_magnitude)
    thresh1_mean = grad_mean + 0.5 * grad_std
    thresh2_mean = grad_mean + 1.5 * grad_std
    
    # Method 3: Otsu's method adapted for gradients
    # Use histogram-based approach
    hist, bins = np.histogram(gradient_magnitude, bins=256, range=(0, 255))
    total_pixels = gradient_magnitude.size
    
    # Find optimal threshold using between-class variance
    current_max = 0
    threshold_otsu = 0
    
    for t in range(1, 256):
        # Background and foreground weights
        w_bg = np.sum(hist[:t]) / total_pixels
        w_fg = np.sum(hist[t:]) / total_pixels
        
        if w_bg == 0 or w_fg == 0:
            continue
            
        # Background and foreground means
        mean_bg = np.sum(np.arange(t) * hist[:t]) / np.sum(hist[:t]) if np.sum(hist[:t]) > 0 else 0
        mean_fg = np.sum(np.arange(t, 256) * hist[t:]) / np.sum(hist[t:]) if np.sum(hist[t:]) > 0 else 0
        
        # Between-class variance
        var_between = w_bg * w_fg * (mean_bg - mean_fg) ** 2
        
        if var_between > current_max:
            current_max = var_between
            threshold_otsu = t
    
    thresh1_otsu = threshold_otsu * 0.5
    thresh2_otsu = threshold_otsu * 1.5
    
    # Method 4: Adaptive based on image content
    # Use local statistics
    thresh1_adaptive = np.percentile(gradient_magnitude[gradient_magnitude > 0], 60)
    thresh2_adaptive = np.percentile(gradient_magnitude[gradient_magnitude > 0], 90)
    
    methods = {
        'Percentile (50/85)': (int(thresh1_p50), int(thresh2_p85)),
        'Mean + Std': (int(thresh1_mean), int(thresh2_mean)),
        'Otsu-based': (int(thresh1_otsu), int(thresh2_otsu)),
        'Adaptive (60/90)': (int(thresh1_adaptive), int(thresh2_adaptive)),
        'Manual (50/150)': (50, 150)
    }
    
    if visualize:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # Original phase
        axes[0, 0].imshow(phase_norm, cmap='gray')
        axes[0, 0].set_title('Original Phase (Normalized)')
        axes[0, 0].axis('off')
        
        # Gradient magnitude distribution
        axes[0, 1].hist(gradient_magnitude.ravel(), bins=50, alpha=0.7, color='blue')
        axes[0, 1].set_title('Gradient Magnitude Distribution')
        axes[0, 1].set_xlabel('Gradient Magnitude')
        axes[0, 1].set_ylabel('Frequency')
        axes[0, 1].grid(True, alpha=0.3)
        
        # Add threshold lines for different methods
        colors = ['red', 'green', 'orange', 'purple', 'brown']
        for i, (method, (t1, t2)) in enumerate(methods.items()):
            axes[0, 1].axvline(t1, color=colors[i], linestyle='--', alpha=0.7, label=f'{method} T1')
            axes[0, 1].axvline(t2, color=colors[i], linestyle='-', alpha=0.7, label=f'{method} T2')
        axes[0, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Edge detection results for different methods
        method_names = list(methods.keys())
        positions = [(0, 2), (1, 0), (1, 1), (1, 2)]
        
        for i, pos in enumerate(positions):
            if i < len(method_names):
                method_name = method_names[i]
                t1, t2 = methods[method_name]
                
                # Apply edge detection
                edges = cv2.Canny(phase_norm, t1//2, t2//2)
                
                axes[pos].imshow(edges, cmap='gray')
                axes[pos].set_title(f'{method_name}\nT1={t1}, T2={t2}')
                axes[pos].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        print("\nThreshold Calculation Methods Comparison:")
        print("=" * 50)
        for method, (t1, t2) in methods.items():
            print(f"{method:20s}: T1={t1:3d}, T2={t2:3d}, Ratio={t2/t1 if t1 > 0 else 'inf':4.1f}")
        
        print(f"\nGradient Statistics:")
        print(f"Mean: {grad_mean:.2f}")
        print(f"Std:  {grad_std:.2f}")
        print(f"Min:  {gradient_magnitude.min():.2f}")
        print(f"Max:  {gradient_magnitude.max():.2f}")
        print(f"Otsu threshold: {threshold_otsu}")
    
    return methods, gradient_magnitude

print("Comprehensive threshold analysis function defined successfully!")

In [ ]:
# Run the comprehensive threshold analysis
print("Running comprehensive threshold analysis...")
threshold_methods, grad_mag = analyze_threshold_methods(Re, Im, visualize=True)

# Apply the improved hyperbola analysis with adaptive thresholds
print("\nApplying improved hyperbola analysis with adaptive thresholds...")
hyperbola_location_adaptive, processed_phase_adaptive, thresholds_used = hyperbola_sar_analysis_adaptive(
    Re, Im, adaptive_thresholds=True, visualize=True
)

print(f"\nResults:")
print(f"Adaptive thresholds used: threshold1={thresholds_used[0]}, threshold2={thresholds_used[1]}")
print(f"Hyperbola-fitted scatterer location: (x={hyperbola_location_adaptive[0]:.1f}, y={hyperbola_location_adaptive[1]:.1f})")

# Compare with manual thresholds for reference
print(f"\nComparison with manual thresholds (50, 150):")
hyperbola_location_manual, _, manual_thresholds = hyperbola_sar_analysis_adaptive(
    Re, Im, threshold1=50, threshold2=150, adaptive_thresholds=False, visualize=False
)
print(f"Manual threshold location: (x={hyperbola_location_manual[0]:.1f}, y={hyperbola_location_manual[1]:.1f})")

# Calculate difference
diff_x = abs(hyperbola_location_adaptive[0] - hyperbola_location_manual[0])
diff_y = abs(hyperbola_location_adaptive[1] - hyperbola_location_manual[1])
print(f"Difference: Δx={diff_x:.1f}, Δy={diff_y:.1f}")

# Test all threshold methods
print(f"\nTesting all threshold methods:")
print("=" * 50)
for method_name, (t1, t2) in threshold_methods.items():
    location, _, thresholds = hyperbola_sar_analysis_adaptive(
        Re, Im, threshold1=t1, threshold2=t2, adaptive_thresholds=False, visualize=False
    )
    print(f"{method_name:20s}: Location=(x={location[0]:6.1f}, y={location[1]:6.1f})")

In [ ]:
# Test the best performing method based on edge quality
def evaluate_edge_quality(edges, phase_norm):
    """
    Evaluate the quality of edge detection results.
    Returns a score based on edge continuity and density.
    """
    # Count edge pixels
    edge_density = np.sum(edges > 0) / edges.size
    
    # Evaluate edge continuity using connected components
    num_labels, labels = cv2.connectedComponents(edges)
    
    # Calculate average component size (excluding background)
    if num_labels > 1:
        component_sizes = []
        for label in range(1, num_labels):
            component_size = np.sum(labels == label)
            component_sizes.append(component_size)
        avg_component_size = np.mean(component_sizes) if component_sizes else 0
        continuity_score = avg_component_size / (phase_norm.size / 1000)  # Normalize
    else:
        continuity_score = 0
    
    # Combined score (balance between density and continuity)
    quality_score = edge_density * 0.3 + min(continuity_score, 1.0) * 0.7
    
    return quality_score, edge_density, continuity_score, num_labels-1

# Evaluate all threshold methods
print("\nEvaluating Edge Detection Quality:")
print("=" * 60)

quality_results = {}
for method_name, (t1, t2) in threshold_methods.items():
    # Apply edge detection
    phase_norm = ((gaussian_filter(np.angle(Re + 1j * Im), sigma=1.5) - 
                   gaussian_filter(np.angle(Re + 1j * Im), sigma=1.5).min()) / 
                  (gaussian_filter(np.angle(Re + 1j * Im), sigma=1.5).max() - 
                   gaussian_filter(np.angle(Re + 1j * Im), sigma=1.5).min()) * 255).astype(np.uint8)
    
    edges = cv2.Canny(phase_norm, t1//2, t2//2)
    
    # Evaluate quality
    quality_score, edge_density, continuity_score, num_components = evaluate_edge_quality(edges, phase_norm)
    
    quality_results[method_name] = {
        'quality_score': quality_score,
        'edge_density': edge_density,
        'continuity_score': continuity_score,
        'num_components': num_components,
        'thresholds': (t1, t2)
    }
    
    print(f"{method_name:20s}: Quality={quality_score:.3f}, Density={edge_density:.4f}, "
          f"Continuity={continuity_score:.3f}, Components={num_components}")

# Find the best method
best_method = max(quality_results.keys(), key=lambda k: quality_results[k]['quality_score'])
best_thresholds = quality_results[best_method]['thresholds']

print(f"\nBest performing method: {best_method}")
print(f"Optimal thresholds: T1={best_thresholds[0]}, T2={best_thresholds[1]}")
print(f"Quality score: {quality_results[best_method]['quality_score']:.3f}")

# Apply the best method to the hyperbola analysis
print(f"\nApplying best method to hyperbola analysis...")
best_location, _, _ = hyperbola_sar_analysis_adaptive(
    Re, Im, 
    threshold1=best_thresholds[0], 
    threshold2=best_thresholds[1], 
    adaptive_thresholds=False, 
    visualize=True
)

print(f"Optimized scatterer location: (x={best_location[0]:.1f}, y={best_location[1]:.1f})")

# Summary comparison
print(f"\nSummary Comparison:")
print(f"=" * 40)
print(f"{'Method':<20} {'X Location':<12} {'Y Location':<12}")
print(f"{'-'*20} {'-'*12} {'-'*12}")
print(f"{'Adaptive':<20} {hyperbola_location_adaptive[0]:<12.1f} {hyperbola_location_adaptive[1]:<12.1f}")
print(f"{'Manual (50,150)':<20} {hyperbola_location_manual[0]:<12.1f} {hyperbola_location_manual[1]:<12.1f}")
print(f"{'Best Quality':<20} {best_location[0]:<12.1f} {best_location[1]:<12.1f}")

In [ ]:
""" DEPRECATED: This section is not used in the final code
phase_components = extract_dominant_phase_components(phase, N=35)

# Plot all 35 dominant phase components (spatial patterns)
n_components = 35
n_rows, n_cols = 7, 5  # 7x5=35

fig, axs = plt.subplots(n_rows, n_cols, figsize=(22, 18), dpi=140)

for i, ((fx, fy), amplitude, spatial_pattern) in enumerate(phase_components[:n_components]):
    ax = axs[i // n_cols, i % n_cols]
    ax.imshow(spatial_pattern, cmap='inferno')
    ax.set_title(f'({fx}, {fy})')
    ax.axis('off')

plt.suptitle('Top 35 Dominant Phase Components (Spatial Patterns)')
plt.tight_layout()
plt.show()"""

# Fringe Method A

In [ ]:
from typing import Tuple, Optional, Union
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
from scipy.signal import hilbert
from skimage.restoration import denoise_tv_chambolle
from skimage.feature import blob_log, peak_local_max


def hilbert_analytic_signal(image: np.ndarray) -> np.ndarray:
    """
    Compute the analytic signal using Hilbert transforms along both axes and return the phase.
    
    This function applies the 2D Hilbert transform to extract the instantaneous phase
    of the input image, which is useful for analyzing fringe patterns in SAR data.
    
    Args:
        image (np.ndarray): Input 2D image array (real-valued).
        
    Returns:
        np.ndarray: Phase angle of the analytic signal in the range [-π, π].
        
    Raises:
        ValueError: If input image is not 2D.
    """
    if image.ndim != 2:
        raise ValueError("Input image must be 2D.")
        
    analytic_signal = hilbert(image, axis=0) + 1j * hilbert(image, axis=1)
    return np.angle(analytic_signal)


def bandpass_filter(phase: np.ndarray, r_in: int = 15, r_out: int = 60) -> np.ndarray:
    """
    Apply a circular bandpass filter in the frequency domain to the phase data.
    
    This filter removes low-frequency trends and high-frequency noise, keeping only
    the intermediate frequencies that typically contain the fringe patterns of interest.
    
    Args:
        phase (np.ndarray): Input phase matrix (2D, real-valued).
        r_in (int, optional): Inner radius of the bandpass filter. Defaults to 15.
        r_out (int, optional): Outer radius of the bandpass filter. Defaults to 60.
        
    Returns:
        np.ndarray: Filtered phase matrix with the same shape as input.
        
    Raises:
        ValueError: If input phase is not 2D or if r_in >= r_out.
    """
    if phase.ndim != 2:
        raise ValueError("Input phase must be 2D.")
    if r_in >= r_out:
        raise ValueError("Inner radius must be smaller than outer radius.")
        
    h, w = phase.shape
    f = np.fft.fftshift(np.fft.fft2(phase))
    H = np.zeros_like(phase)
    cy, cx = h // 2, w // 2
    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X - cx) ** 2 + (Y - cy) ** 2)
    H[(dist >= r_in) & (dist <= r_out)] = 1
    f_filtered = f * H
    return np.angle(np.fft.ifft2(np.fft.ifftshift(f_filtered)))


def normalize_and_equalize(phase_image: np.ndarray) -> np.ndarray:
    """
    Normalize phase image to [0, 255] range and apply histogram equalization.
    
    This function enhances the contrast of the phase image by normalizing its
    dynamic range and applying adaptive histogram equalization.
    
    Args:
        phase_image (np.ndarray): Input phase image (2D, real-valued).
        
    Returns:
        np.ndarray: Contrast-enhanced image as uint8 array.
        
    Raises:
        ValueError: If input image is not 2D.
    """
    if phase_image.ndim != 2:
        raise ValueError("Input phase image must be 2D.")
        
    # Normalize to [0, 255] range
    norm = ((phase_image - phase_image.min()) / 
            (phase_image.max() - phase_image.min()) * 255).astype(np.uint8)
    return cv2.equalizeHist(norm)


def refine_enhancement(image: np.ndarray, tv_weight: float = 0.2, 
                      median_size: int = 5) -> np.ndarray:
    """
    Apply total variation denoising followed by median filtering for image refinement.
    
    This two-step process first removes noise while preserving edges using total
    variation denoising, then applies median filtering to remove remaining outliers.
    
    Args:
        image (np.ndarray): Input image to be refined (2D).
        tv_weight (float, optional): Weight for total variation denoising. Defaults to 0.2.
        median_size (int, optional): Size of the median filter kernel. Defaults to 5.
        
    Returns:
        np.ndarray: Refined image with reduced noise.
        
    Raises:
        ValueError: If input image is not 2D or parameters are invalid.
    """
    if image.ndim != 2:
        raise ValueError("Input image must be 2D.")
    if tv_weight <= 0:
        raise ValueError("TV weight must be positive.")
    if median_size <= 0 or median_size % 2 == 0:
        raise ValueError("Median size must be a positive odd integer.")
        
    tv_denoised = denoise_tv_chambolle(image.astype(np.float32), weight=tv_weight)
    return median_filter(tv_denoised, size=median_size)


def estimate_scatterer_location(enhanced_image: np.ndarray, 
                               initial_guess: Optional[Tuple[int, int]] = None,
                               min_sigma: float = 3, max_sigma: float = 20,
                               threshold: float = 0.02) -> Tuple[int, int]:
    """
    Estimate the location of scatterers in the enhanced image using blob detection.
    
    This function uses either blob detection for initial estimation or local peak
    detection for refinement when an initial guess is provided.
    
    Args:
        enhanced_image (np.ndarray): Input enhanced image (2D).
        initial_guess (Optional[Tuple[int, int]], optional): Initial guess for refinement.
            If None, performs initial blob detection. Defaults to None.
        min_sigma (float, optional): Minimum blob size for detection. Defaults to 3.
        max_sigma (float, optional): Maximum blob size for detection. Defaults to 20.
        threshold (float, optional): Detection threshold. Defaults to 0.02.
        
    Returns:
        Tuple[int, int]: Estimated (y, x) coordinates of the scatterer.
        
    Raises:
        ValueError: If no blobs/peaks are detected or input image is not 2D.
    """
    if enhanced_image.ndim != 2:
        raise ValueError("Input enhanced image must be 2D.")
        
    if initial_guess is None:
        # Initial estimation using blob detection
        blobs = blob_log(enhanced_image, min_sigma=min_sigma, max_sigma=max_sigma, 
                        num_sigma=10, threshold=threshold)
        if len(blobs) == 0:
            raise ValueError("No blobs detected. Try adjusting threshold or sigma parameters.")
        blobs = blobs[np.argsort(-blobs[:, 2])]  # Sort by strength (descending)
        return int(blobs[0][0]), int(blobs[0][1])
    else:
        # Refinement using local peak detection
        local_peaks = peak_local_max(enhanced_image, min_distance=10, threshold_abs=0.1)
        if len(local_peaks) == 0:
            raise ValueError("No local peaks detected for refinement.")
        dists = np.linalg.norm(local_peaks - np.array(initial_guess), axis=1)
        closest_peak = local_peaks[np.argmin(dists)]
        return tuple(closest_peak)


def process_sar_fringe_pattern(real_part: np.ndarray, 
                              imaginary_part: np.ndarray,
                              r_in: int = 15, 
                              r_out: int = 60,
                              sigma: float = 1.0,
                              visualize: bool = True) -> Tuple[Tuple[int, int], np.ndarray]:
    """
    Complete processing pipeline for SAR fringe pattern analysis and scatterer detection.
    
    This function implements a comprehensive workflow for analyzing SAR interferometric
    data to detect and locate scatterers through fringe pattern analysis.
    
    Args:
        real_part (np.ndarray): Real component of the complex SAR image.
        imaginary_part (np.ndarray): Imaginary component of the complex SAR image.
        r_in (int, optional): Inner radius for bandpass filter. Defaults to 15.
        r_out (int, optional): Outer radius for bandpass filter. Defaults to 60.
        sigma (float, optional): Gaussian smoothing parameter. Defaults to 1.0.
        visualize (bool, optional): Whether to display results. Defaults to True.
        
    Returns:
        Tuple[Tuple[int, int], np.ndarray]: 
            - Refined scatterer location as (y, x) coordinates
            - Smoothed input channel for visualization
            
    Raises:
        ValueError: If input arrays have different shapes or are not 2D.
    """
    if real_part.shape != imaginary_part.shape:
        raise ValueError("Real and imaginary parts must have the same shape.")
    if real_part.ndim != 2:
        raise ValueError("Input arrays must be 2D.")
        
    # Create smoothed channel from real part (assuming this was the intended input)
    from scipy.ndimage import gaussian_filter
    smoothed_ch = gaussian_filter(real_part, sigma=sigma)
    
    # Apply the fringe analysis pipeline
    phase_ch = hilbert_analytic_signal(smoothed_ch)
    filtered_phase = bandpass_filter(phase_ch, r_in=r_in, r_out=r_out)
    enhanced = normalize_and_equalize(filtered_phase)
    cleaned = refine_enhancement(enhanced)
    
    # Estimate scatterer location with refinement
    init_y, init_x = estimate_scatterer_location(cleaned)
    refined_y, refined_x = estimate_scatterer_location(cleaned, 
                                                      initial_guess=(init_y, init_x))
    
    if visualize:
        plt.figure(figsize=(12, 4))
        
        # Show original smoothed channel
        plt.subplot(1, 3, 1)
        plt.imshow(smoothed_ch, cmap='gray')
        plt.title("Smoothed Real Component")
        plt.axis('off')
        
        # Show enhanced phase
        plt.subplot(1, 3, 2) 
        plt.imshow(cleaned, cmap='gray')
        plt.title("Enhanced Phase")
        plt.axis('off')
        
        # Show result with detected scatterer
        plt.subplot(1, 3, 3)
        plt.imshow(smoothed_ch, cmap='gray')
        plt.scatter(refined_x, refined_y, c='lime', s=100, marker='x', 
                   label='Refined Scatterer', linewidths=3)
        plt.title("Detected Scatterer Location")
        plt.legend()
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
    return (refined_y, refined_x), smoothed_ch


# --------------- Main Processing Pipeline Execution ---------------

# Process the SAR data using the available Re and Im components
scatterer_location, processed_channel = process_sar_fringe_pattern(
    real_part=Re, 
    imaginary_part=Im,
    r_in=15,
    r_out=160,
    sigma=.5,
    visualize=True
)

print(f"Detected scatterer at coordinates: (y={scatterer_location[0]}, x={scatterer_location[1]})")


📌 Use Cases
	•	Fringe pattern orientation analysis.
	•	Directional filtering (e.g., Hough-style for SAR fringes).
	•	Initial phase ramp estimation for correction in InSAR.

Let me know if you want the inverse Radon (reconstruction) or directional filtering based on Radon peaks.

In [ ]:
import numpy as np
from skimage.transform import radon

def compute_radon_transform(phase_matrix: np.ndarray, theta: np.ndarray = None):
    """
    Compute the Radon transform of a 2D wrapped phase matrix.

    Args:
        phase_matrix (np.ndarray): 2D input phase matrix (real-valued).
        theta (np.ndarray, optional): Array of projection angles (degrees). 
            Defaults to np.linspace(0., 180., max(phase_matrix.shape), endpoint=False).

    Returns:
        radon_image (np.ndarray): 2D Radon transform (ρ × θ).
        theta (np.ndarray): Array of angles used.
    """
    if theta is None:
        theta = np.linspace(0., 180., max(phase_matrix.shape), endpoint=False)

    radon_image = radon(phase_matrix, theta=theta, circle=False)
    return radon_image, theta

In [ ]:
# Plot only the real and imaginary parts
fig, axs = plt.subplots(1, 2, figsize=(10, 5), dpi=140)

axs[0].imshow(Re, cmap='inferno')
axs[0].set_title('Real Part')
axs[0].axis('off')

axs[1].imshow(Im, cmap='inferno')
axs[1].set_title('Imaginary Part')
axs[1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Plot the difference between the real and imaginary parts
diff = Re - Im

plt.figure(figsize=(6, 5), dpi=140)
plt.imshow(diff, cmap='inferno')
plt.title('Difference: Real - Imaginary')
plt.axis('off')
plt.colorbar()
plt.show()

# 3D surface plot of the normalized difference
diff_norm = (diff - np.mean(diff)) / np.std(diff)  # z-score normalization


X = np.arange(diff_norm.shape[1])
Y = np.arange(diff_norm.shape[0])
X, Y = np.meshgrid(X, Y)


fig = go.Figure(data=[go.Surface(z=diff_norm, x=X, y=Y, colorscale='RdBu', cmin=-1, cmax=1, colorbar=dict(title='z-score'))])
fig.update_layout(
    title='3D Surface: Normalized Difference (z-score)',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='z-score'
    ),
    autosize=False,
    width=800,
    height=600,
    margin=dict(l=65, r=50, b=65, t=90)
)
fig.show()




In [ ]:
# 3D surface plot of the phase (z-score normalized)
phase_norm = (phase - np.mean(phase)) / np.std(phase)

fig_phase = go.Figure(data=[go.Surface(z=phase_norm, x=X, y=Y, colorscale='Viridis', colorbar=dict(title='z-score'))])
fig_phase.update_layout(
    title='3D Surface: Normalized Phase (z-score)',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='z-score'
    ),
    autosize=False,
    width=1800,
    height=1600,
    margin=dict(l=65, r=50, b=65, t=90)
)
fig_phase.show()

In [ ]:
train_loader = dm.train_dataloader()
batch = next(iter(train_loader))
print("Batch keys:", batch.keys() if hasattr(batch, 'keys') else type(batch))
print("Batch shapes:")
for k, v in batch.items():
    print(f"{k}: {v.shape}" if hasattr(v, 'shape') else f"{k}: {type(v)}")

In [ ]:
# Configuration Parameters (Replaces argparse and config.ini for notebook usage)

# Simulating command line arguments (set manually for debugging)
class Args:
    def __init__(self):
        self.gen = None  # Or specify a generation number, e.g., 1, 2, ...
        # self.config = 'config.ini' # config.ini will be read effectively by hardcoding params below

args = Args()

# Simulating config.ini parameters (set manually for debugging)
# These values would typically be read from config.ini
config_params = {
    'Computation': {
        'seed': 42  # Default seed
    },
    'NAS': {
        'max_layers': 10
    },
    'GA': {
        'max_iterations': 50,
        'population_size': 20,
        'mating_pool_cutoff': 0.5,
        'mutation_probability': 0.2,
        'epochs': 10,
        'batch_size': 4, # Note: dm also has batch_size, ensure consistency or decide which one to use
        'n_random': 5,
        'k_best': 5
    }
}

# Apply seed and precision from config
pl.seed_everything(seed=config_params['Computation']['seed'], workers=True)  # For reproducibility
torch.set_float32_matmul_precision("medium")  # to make lightning happy

print("Configuration parameters set.")
print(f"Generation to load: {args.gen}")
print(f"Seed: {config_params['Computation']['seed']}")

In [ ]:
# Main Function Definition
def main_notebook(notebook_args, notebook_config, datamodule):
    # Seed and precision are set in the cell above based on notebook_config

    # Model parameters from notebook_config
    max_layers = int(notebook_config['NAS']['max_layers'])
    max_gen = int(notebook_config['GA']['max_iterations'])
    n_individuals = int(notebook_config['GA']['population_size'])
    mating_pool_cutoff = float(notebook_config['GA']['mating_pool_cutoff'])
    mutation_probability = float(notebook_config['GA']['mutation_probability'])
    epochs = int(notebook_config['GA']['epochs'])
    batch_size = int(notebook_config['GA']['batch_size']) # Ensure this aligns with dm or is used consistently
    n_random = int(notebook_config['GA']['n_random'])
    k_best = int(notebook_config['GA']['k_best'])
    
    print("Initializing Population...")
    # Define population
    pop = Population(n_individuals=n_individuals, max_layers=max_layers, dm=datamodule, max_parameters=400_000)
    
    if notebook_args.gen is not None:
        print(f"Loading generation: {notebook_args.gen}")
        pop.load_generation(notebook_args.gen)
    else:
        print("Performing initial poll for population...")
        pop.initial_poll()

    print(f"Starting training for {max_gen} generations...")
    for gen_idx in range(max_gen):
        print(f"--- Generation {gen_idx + 1}/{max_gen} ---")
        print("Training generation...")
        pop.train_generation(task='classification', lr=0.001, epochs=epochs, batch_size=batch_size)
        print("Evolving population...")
        pop.evolve(mating_pool_cutoff=mating_pool_cutoff, mutation_probability=mutation_probability, k_best=k_best, n_random=n_random)
        print(f"--- End of Generation {gen_idx + 1} ---")
        
    print("Process finished.")

print("Main function 'main_notebook' defined.")

In [ ]:
# Call the Main Function
# This replaces the if __name__ == '__main__': block

# Ensure 'dm' (DataModule), 'args' (simulated command-line args), 
# and 'config_params' (simulated config.ini) are defined from previous cells.

print("Running main_notebook function...")
main_notebook(args, config_params, dm)
print("Notebook execution complete.")